# 📈 ML Pipeline Prototyping - Sales Forecasting using XGBoost

This notebook demonstrates the time-series forecasting prototype for the **Sales Forecasting Dashboard**. 
It loads data from our live **Neon PostgreSQL cloud database**, performs data imputation, engineers advanced features (lags, rolling averages, temporal seasonality), trains an **XGBoost Regressor**, and validates accuracy using **Mean Absolute Percentage Error (MAPE)**.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import timedelta
from sqlalchemy import create_engine
from sklearn.metrics import mean_absolute_error

# Add backend directory to system path to import settings
sys.path.append(os.path.abspath('backend'))
from app.core.config import get_settings

settings = get_settings()
print("Settings loaded successfully.")
print(f"Database Host: {settings.POSTGRES_SERVER}")

Settings loaded successfully.
Database Host: ep-royal-rice-avd12cx6.c-11.us-east-1.aws.neon.tech


## 🗄️ 1. Extract Data from Neon Database

In [2]:
# Connect to Neon Postgres using the sync connection string (psycopg2)
engine = create_engine(settings.DATABASE_URL_SYNC)

# Load products and sales history
df_products = pd.read_sql("SELECT * FROM products", engine)
df_sales = pd.read_sql("SELECT * FROM sales_history", engine)

print(f"Loaded {len(df_products)} products.")
print(f"Loaded {len(df_sales)} sales history records.")
df_sales.head()

Loaded 3 products.
Loaded 2922 sales history records.


,id,product_id,date,units_sold,revenue
0,1,711b8bc8-989c-4ceb-99f5-093b4699557e,2024-01-01,25,3750.0
1,2,711b8bc8-989c-4ceb-99f5-093b4699557e,2024-01-02,21,3150.0
2,3,711b8bc8-989c-4ceb-99f5-093b4699557e,2024-01-03,20,3000.0
3,4,711b8bc8-989c-4ceb-99f5-093b4699557e,2024-01-04,22,3300.0
4,5,711b8bc8-989c-4ceb-99f5-093b4699557e,2024-01-05,19,2850.0


## 🔄 2. Data Imputation
Sales data is rarely continuous. We must ensure every product has a record for every single date between the minimum and maximum dates, filling missing dates with `0` units sold.

In [3]:
df_sales['date'] = pd.to_datetime(df_sales['date'])

imputed_dfs = []
min_date = df_sales['date'].min()
max_date = df_sales['date'].max()
full_date_range = pd.date_range(start=min_date, end=max_date, freq='D')

for product_id in df_sales['product_id'].unique():
    # Create complete date series for this product
    product_full = pd.DataFrame({
        'date': full_date_range,
        'product_id': product_id
    })
    
    # Merge with actual sales records
    prod_sales = df_sales[df_sales['product_id'] == product_id]
    merged = pd.merge(product_full, prod_sales, on=['product_id', 'date'], how='left')
    
    # Impute missing values (0 units sold, 0.00 revenue)
    merged['units_sold'] = merged['units_sold'].fillna(0).astype(int)
    merged['revenue'] = merged['revenue'].fillna(0.0).astype(float)
    merged['id'] = merged['id'].fillna(-1).astype(int)
    
    imputed_dfs.append(merged)

df_clean = pd.concat(imputed_dfs, ignore_index=True)
df_clean = df_clean.sort_values(by=['product_id', 'date']).reset_index(drop=True)
print(f"Data imputed. Total rows: {len(df_clean)}")
df_clean.head()

Data imputed. Total rows: 2922


,date,product_id,id,units_sold,revenue
0,2024-01-01,1fad6a2a-704e-45c0-812c-962609d9806b,1949,11,495.0
1,2024-01-02,1fad6a2a-704e-45c0-812c-962609d9806b,1950,11,495.0
2,2024-01-03,1fad6a2a-704e-45c0-812c-962609d9806b,1951,15,675.0
3,2024-01-04,1fad6a2a-704e-45c0-812c-962609d9806b,1952,13,585.0
4,2024-01-05,1fad6a2a-704e-45c0-812c-962609d9806b,1953,12,540.0


## 🧠 3. Feature Engineering
We create Lags, Rolling Averages, and Temporal Features to help XGBoost predict time-series dependencies.

In [4]:
# A. Lag Features (yesterday, last week, last month)
df_clean['sales_lag_1'] = df_clean.groupby('product_id')['units_sold'].shift(1)
df_clean['sales_lag_7'] = df_clean.groupby('product_id')['units_sold'].shift(7)
df_clean['sales_lag_30'] = df_clean.groupby('product_id')['units_sold'].shift(30)

# B. Rolling Statistics (7-day mean, 30-day standard deviation)
df_clean['sales_roll_mean_7'] = df_clean.groupby('product_id')['units_sold'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=1).mean()
)
df_clean['sales_roll_std_30'] = df_clean.groupby('product_id')['units_sold'].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=1).std()
)

# C. Temporal Features
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek
df_clean['is_weekend'] = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['month'] = df_clean['date'].dt.month

# Simple holiday helper (November / December peak holiday periods)
df_clean['is_holiday'] = ((df_clean['month'] == 11) & (df_clean['date'].dt.day >= 20) | 
                          (df_clean['month'] == 12)).astype(int)

# Remove rows containing NaN caused by shifting
df_features = df_clean.dropna().reset_index(drop=True)
print(f"Feature matrix built. Final shape: {df_features.shape}")
df_features.head()

Feature matrix built. Final shape: (2832, 14)


,date,product_id,id,units_sold,revenue,sales_lag_1,sales_lag_7,sales_lag_30,sales_roll_mean_7,sales_roll_std_30,day_of_week,is_weekend,month,is_holiday
0,2024-01-31,1fad6a2a-704e-45c0-812c-962609d9806b,1979,13,585.0,15.0,15.0,11.0,16.142857,3.268801,2,0,1,0
1,2024-02-01,1fad6a2a-704e-45c0-812c-962609d9806b,1980,14,630.0,13.0,12.0,11.0,15.857143,3.205599,3,0,2,0
2,2024-02-02,1fad6a2a-704e-45c0-812c-962609d9806b,1981,11,495.0,14.0,15.0,15.0,16.142857,3.122223,4,0,2,0
3,2024-02-03,1fad6a2a-704e-45c0-812c-962609d9806b,1982,21,945.0,11.0,19.0,13.0,15.571429,3.210794,5,1,2,0
4,2024-02-04,1fad6a2a-704e-45c0-812c-962609d9806b,1983,21,945.0,21.0,23.0,12.0,15.857143,3.370085,6,1,2,0


## 📊 4. Train / Test Split
Never use random splits for time-series forecasting. We split sequentially: train on the past, test on the future (using the last 30 days as validation test set).

In [5]:
split_date = df_features['date'].max() - timedelta(days=30)

train_df = df_features[df_features['date'] <= split_date]
test_df = df_features[df_features['date'] > split_date]

features = [
    'sales_lag_1', 'sales_lag_7', 'sales_lag_30', 
    'sales_roll_mean_7', 'sales_roll_std_30',
    'day_of_week', 'is_weekend', 'month', 'is_holiday'
]
target = 'units_sold'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

print(f"Training set rows: {len(X_train)} (Up to {split_date.strftime('%Y-%m-%d')})")
print(f"Testing set rows: {len(X_test)} (After {split_date.strftime('%Y-%m-%d')})")

Training set rows: 2742 (Up to 2026-08-01)
Testing set rows: 90 (After 2026-08-01)


## 🤖 5. Train XGBoost Model

In [6]:
model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)
print("XGBoost Regressor model trained successfully!")

XGBoost Regressor model trained successfully!


## 📈 6. Evaluation and MAPE Metric Calculation

In [7]:
predictions = model.predict(X_test)

# Ensure no negative sales are predicted
predictions = np.clip(predictions, 0, None)

# Calculate MAPE (excluding days where actual sales were zero to avoid division by zero)
actuals = y_test.values
mask = actuals > 0

mape = np.mean(np.abs((actuals[mask] - predictions[mask]) / actuals[mask])) * 100
mae = mean_absolute_error(actuals, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} units")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

Mean Absolute Error (MAE): 3.10 units
Mean Absolute Percentage Error (MAPE): 11.77%
